In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
key = pd.read_csv('https://raw.githubusercontent.com/HSEAi2025Team-7/keyRateForecast/Liza/key_inflation.csv')
key['date'] = key['date'].astype(str)
n = 13
for i in range(57, 69):
    n -= 1
    key.loc[i, 'date'] = f'{n}.2020'
# переводим даты в формат дат
key["date"] = pd.to_datetime(key["date"], format="%m.%Y")
key['key-rate'] = key['key-rate'].replace(',', '.', regex=True).astype(np.float64)
key['inflation'] = key['inflation'].replace(',', '.', regex=True).astype(np.float64)

In [ ]:
# добавляем колонку цель по инфляции = 4
key['inflation_target'] = 4
# добавляем колонку разница между инфляцией и целью по инфляции
key['difference'] = key['inflation'] - key['inflation_target']
# добавляем колонку разница межды инфляцией
key['inflation_diff'] = key['inflation'].diff().shift(-1)
# заполняем пропуск медианой
med_inflation_diff = key["inflation_diff"].median()
key["inflation_diff"] = key["inflation_diff"].fillna(med_inflation_diff)
# прошлая ключевая ставка
key['key_rate_last'] = key['key-rate'].shift(-1)
med_key_rate_last = key['key_rate_last'].median()
key['key_rate_last'] = key['key_rate_last'].fillna(med_key_rate_last)
# на сколько изменилась ставка
key['key_rate_last_diff'] = key['key_rate_last'].diff().shift(-1)
med_key_rate_diff = key["key_rate_last_diff"].median()
key["key_rate_last_diff"] = key["key_rate_last_diff"].fillna(med_key_rate_diff)
# добавляем колонку изменение ключевой ставки
key['change_key_rate'] = np.sign(key['key-rate'].diff()).astype('Int64').shift(-1)
med_change_key_rate = key["change_key_rate"].median()
key["change_key_rate"] = key["change_key_rate"].fillna(med_change_key_rate)

In [ ]:
key.isna().sum()

,0
date,0
key-rate,0
inflation,0
inflation_target,0
difference,0
inflation_diff,0
key_rate_last,0
key_rate_last_diff,0
change_key_rate,0


In [ ]:
dollar = pd.read_csv('https://raw.githubusercontent.com/HSEAi2025Team-7/keyRateForecast/Liza/currency_rate.csv')
dollar["date"] = pd.to_datetime(dollar["date"], format="%d.%m.%Y")
dollar['dollar_rate'] = dollar['dollar_rate'].replace(',', '.', regex=True).astype(np.float64)
dollar_month = (dollar.set_index('date').resample('MS')['dollar_rate'].first().reset_index())
key = key.merge(dollar_month, on='date', how='left')

In [ ]:
key.head(20)

,date,key-rate,inflation,inflation_target,difference,inflation_diff,key_rate_last,key_rate_last_diff,change_key_rate
0,2025-09-01,17.0,7.98,4,3.98,0.16,18.0,0.0,1
1,2025-08-01,18.0,8.14,4,4.14,0.65,18.0,2.0,0
2,2025-07-01,18.0,8.79,4,4.79,0.61,20.0,1.0,1
3,2025-06-01,20.0,9.40,4,5.40,0.48,21.0,0.0,1
4,2025-05-01,21.0,9.88,4,5.88,0.35,21.0,0.0,0
5,2025-04-01,21.0,10.23,4,6.23,0.11,21.0,0.0,0
6,2025-03-01,21.0,10.34,4,6.34,-0.28,21.0,0.0,0
7,2025-02-01,21.0,10.06,4,6.06,-0.14,21.0,0.0,0
8,2025-01-01,21.0,9.92,4,5.92,-0.40,21.0,0.0,0
9,2024-12-01,21.0,9.52,4,5.52,-0.64,21.0,0.0,0


In [ ]:
# делим на матрицу признаков и целевую переменную
X1 = key[['inflation', 'inflation_target', 'difference', 'inflation_diff', 'key_rate_last', 'key_rate_last_diff', 'dollar_rate']]
y1 = key['change_key_rate']

In [ ]:
# делим на матрицу признаков и целевую переменную
X2 = key[['inflation', 'inflation_target', 'difference', 'inflation_diff', 'key_rate_last', 'key_rate_last_diff', 'dollar_rate']]
y2 = key['key-rate']

In [ ]:
X1.head()

,inflation,inflation_target,difference,inflation_diff,key_rate_last,key_rate_last_diff,dollar_rate
0,7.98,4,3.98,0.16,18.0,0.0,80.4261
1,8.14,4,4.14,0.65,18.0,2.0,80.3163
2,8.79,4,4.79,0.61,20.0,1.0,78.5284
3,9.40,4,5.40,0.48,21.0,0.0,79.1285
4,9.88,4,5.88,0.35,21.0,0.0,81.4933


In [ ]:
y1.head()

,change_key_rate
0,1
1,0
2,1
3,1
4,0


Рабочая модель

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, accuracy_score

# Разделение данных
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.20, random_state=42 #, stratify=y
)

# Масштабирование
scaler = StandardScaler()
X1_train_scaled = scaler.fit_transform(X1_train)
X1_test_scaled = scaler.transform(X1_test)

# Создание модели с фиксированными параметрами
model = LogisticRegression(
    class_weight='balanced',
    penalty= 'l1',#'l1'
    solver= 'liblinear', #'saga'
)

# Обучение модели
model.fit(X1_train_scaled, y1_train)

# Оценка модели на тесте

y1_pred = model.predict(X1_test_scaled)

In [ ]:
print("Accuracy:", accuracy_score(y1_test, y1_pred))
print("f1:", f1_score(y1_test, y1_pred, average='micro'))
print("Отчёт классификации:\n", classification_report(y1_test, y1_pred, target_names=['Понижение ставки', 'Сохраняется','Повышение ставки']))
print("Матрица ошибок:\n", confusion_matrix(y1_test, y1_pred))

Accuracy: 0.5517241379310345
f1: 0.5517241379310345
Отчёт классификации:
                   precision    recall  f1-score   support

Понижение ставки       0.20      0.20      0.20         5
     Сохраняется       0.61      0.78      0.68        18
Повышение ставки       1.00      0.17      0.29         6

        accuracy                           0.55        29
       macro avg       0.60      0.38      0.39        29
    weighted avg       0.62      0.55      0.52        29

Матрица ошибок:
 [[ 1  4  0]
 [ 4 14  0]
 [ 0  5  1]]


In [ ]:
print(y1_pred)
print(y1_test)

[ 0.  0. -1.  0. -1.  0.  1.  0.  0.  0.  0. -1. -1.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0. -1.  0.  0.]
69      1
140     0
27      0
19      0
42      0
117     0
126     1
108     1
84     -1
18      0
12     -1
55      0
128     0
78      0
73      0
36      1
112     0
133     0
100     1
101     0
94      0
136     0
11     -1
66      0
31      0
45     -1
51     -1
76      0
111     1
Name: change_key_rate, dtype: Int64


In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error as MSE

# your code here
# Разделение данных
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42#, stratify=y
    )

    # Масштабирование
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

    # Создание модели с фиксированными параметрами
model = LinearRegression()

                # Обучение модели
model.fit(X2_train_scaled, y2_train)

                # Оценка модели на тесте

y2_pred = model.predict(X2_test_scaled)

In [ ]:
mse = mean_squared_error(y2_test, y2_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y2_test, y2_pred)
mape = np.mean(np.abs((y2_test - y2_pred) / y2_test)) * 100  # в процентах
R2 = r2_score(y2_test, y2_pred)
# Вывод результатов
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"MAPE: {mape:.6f}%")
print(f"R2:   {R2:.6f}")

MSE:  2.423472
RMSE: 1.556750
MAE:  0.734583
MAPE: 5.978656%
R2:   0.883266


In [ ]:
print(y2_pred)
print(y2_test)

[ 6.6546568   5.41614964  8.08848197 15.89696073 26.84193319 10.69170841
 13.95954811 10.37763922  7.57932608 15.86440591 17.57861516  5.10726605
 20.78994609  8.01363466  7.28772944  8.26219892 10.98507178  8.10118002
  9.62261564  9.44659292  8.07584772  7.69318921 19.06119007  6.26140047
  7.83744414  7.99380375  5.77624576  7.94295464 11.0349559 ]
69      6.25
140     5.50
27      7.50
19     16.00
42     20.00
117    11.00
126    14.00
108    10.00
84      7.50
18     16.00
12     19.00
55      4.25
128    17.00
78      7.75
73      7.25
36      7.50
112    11.00
133     8.00
100     9.25
101     9.75
94      8.25
136     7.50
11     21.00
66      6.00
31      7.50
45      8.50
51      5.50
76      7.75
111    10.50
Name: key-rate, dtype: float64
